In [0]:
use catalog identifier(:catalog)

In [0]:
CREATE OR REPLACE TEMP VIEW region_validation AS
SELECT *,
  nullif(concat_ws(',',
    IF(r_regionkey IS NULL, 'NULL_PK', NULL),
    IF(dup_pk, 'DUPLICATE_PK', NULL),
    IF(r_name IS NULL, 'MISSING_NAME', NULL)
  ), '') AS rejection_reason
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY r_regionkey ORDER BY ingestion_ts, file_path) AS row_num_partition,
         (COUNT(*) OVER (PARTITION BY r_regionkey)) > 1 AS dup_pk
  FROM DDEV.bronze.region
);


-- Step 3: Create Silver region
CREATE OR REPLACE TABLE DDEV.silver.region AS
SELECT r_regionkey,
       r_name,
       r_comment
FROM region_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create region quarantine table
CREATE OR REPLACE TABLE DDEV.silver.region_quarantine AS
SELECT r_regionkey,
       r_name,
       r_comment,rejection_reason
FROM region_validation
WHERE rejection_reason IS NOT NULL;


In [0]:
CREATE OR REPLACE TEMP VIEW nation_validation AS
SELECT *,
  nullif(concat_ws(',',
    IF(n_nationkey IS NULL, 'NULL_PK', NULL),
    IF(n_name IS NULL, 'MISSING_NAME', NULL),
    IF(n_regionkey IS NULL, 'MISSING_REGION_PK', NULL),
    IF(n_regionkey NOT IN (SELECT r_regionkey FROM DDEV.silver.region), 'INVALID_REGION_FK', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.nation;

-- Step 3: Create Silver nation
CREATE OR REPLACE TABLE DDEV.silver.nation AS
SELECT * EXCEPT(rejection_reason)
FROM nation_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create nation quarantine table
CREATE OR REPLACE TABLE DDEV.silver.nation_quarantine AS
SELECT *
FROM nation_validation
WHERE rejection_reason IS NOT NULL;


In [0]:
CREATE OR REPLACE TEMP VIEW customer_validation AS
SELECT *,
  nullif(concat_ws(',',
    IF(c_custkey IS NULL, 'NULL_PK', NULL),
    IF(c_name IS NULL, 'MISSING_NAME', NULL),
    IF(c_nationkey IS NULL, 'MISSING_NATION_PK', NULL),
    IF(c_nationkey NOT IN (SELECT n_nationkey FROM DDEV.silver.nation), 'INVALID_NATION_FK', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.customer;

-- Step 3: Create Silver customer
CREATE OR REPLACE TABLE DDEV.silver.customer AS
SELECT c_custkey,
       c_name,
       c_address,
       c_nationkey,
       c_phone,
       c_acctbal,
       c_mktsegment,
       c_comment
FROM customer_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create customer quarantine table
CREATE OR REPLACE TABLE DDEV.silver.customer_quarantine AS
SELECT c_custkey,
       c_name,
       c_address,
       c_nationkey,
       c_phone,
       c_acctbal,
       c_mktsegment,
       c_comment,rejection_reason
FROM customer_validation
WHERE rejection_reason IS NOT NULL;


In [0]:
CREATE OR REPLACE TEMP VIEW supplier_validation AS
SELECT *,
  nullif(concat_ws(',',
    IF(s_suppkey IS NULL, 'NULL_PK', NULL),
    IF(s_name IS NULL, 'MISSING_NAME', NULL),
    IF(s_nationkey IS NULL, 'MISSING_NATION_PK', NULL),
    IF(s_nationkey NOT IN (SELECT n_nationkey FROM DDEV.silver.nation), 'INVALID_NATION_FK', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.supplier;

-- Step 3: Create Silver supplier
CREATE OR REPLACE TABLE DDEV.silver.supplier AS
SELECT s_suppkey,
       s_name,
       s_address,
       s_nationkey,
       s_phone,
       s_acctbal,
       s_comment
FROM supplier_validation
WHERE rejection_reason IS NULL ;

-- Step 4: Create supplier quarantine table
CREATE OR REPLACE TABLE DDEV.silver.supplier_quarantine AS
SELECT s_suppkey,
       s_name,
       s_address,
       s_nationkey,
       s_phone,
       s_acctbal,
       s_comment,rejection_reason
FROM supplier_validation
WHERE rejection_reason IS NOT NULL;



In [0]:
CREATE OR REPLACE TEMP VIEW part_validation AS
SELECT *,
  nullif(concat_ws(',',
    IF(p_partkey IS NULL, 'NULL_PK', NULL),
    IF(p_name IS NULL, 'MISSING_NAME', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.part;

-- Step 3: Create Silver part
CREATE OR REPLACE TABLE DDEV.silver.part AS
SELECT p_partkey,
       p_name,
       p_mfgr,
       p_brand,
       p_type,
       p_size,
       p_container,
       p_retailprice,
       p_comment
FROM part_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create part quarantine table
CREATE OR REPLACE TABLE DDEV.silver.part_quarantine AS
SELECT p_partkey,
       p_name,
       p_mfgr,
       p_brand,
       p_type,
       p_size,
       p_container,
       p_retailprice,
       p_comment,rejection_reason
FROM part_validation
WHERE rejection_reason IS NOT NULL;



In [0]:
CREATE OR REPLACE TEMP VIEW orders_validation AS
SELECT o.*,
  nullif(concat_ws(',',
    IF(o.o_orderkey IS NULL, 'NULL_PK', NULL),
    IF(o.o_custkey IS NULL, 'MISSING_CUST_PK', NULL),
    IF(o.o_custkey IS NOT NULL AND c.c_custkey IS NULL, 'INVALID_CUST_FK', NULL),
    IF(o.o_orderdate IS NULL, 'MISSING_ORDERDATE', NULL),
    IF(o.o_orderdate < '1992-01-01' OR o.o_orderdate > '1998-12-31', 'ORDERDATE_OUT_OF_RANGE', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.orders o
LEFT JOIN (SELECT DISTINCT c_custkey FROM DDEV.silver.customer) c
  ON o.o_custkey = c.c_custkey;

-- Step 3: Create Silver orders
CREATE OR REPLACE TABLE DDEV.silver.orders AS
SELECT o_orderkey,
       o_custkey,
       o_orderstatus,
       o_totalprice,
       o_orderdate,
       o_orderpriority,
       o_clerk,
       o_shippriority,
       o_comment
FROM orders_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create orders quarantine table
CREATE OR REPLACE TABLE DDEV.silver.orders_quarantine AS
SELECT o_orderkey,
       o_custkey,
       o_orderstatus,
       o_totalprice,
       o_orderdate,
       o_orderpriority,
       o_clerk,
       o_shippriority,
       o_comment,rejection_reason
FROM orders_validation
WHERE rejection_reason IS NOT NULL;



In [0]:
CREATE OR REPLACE TEMP VIEW lineitem_validation AS
SELECT l.*,
  nullif(concat_ws(',',
    IF(l.l_orderkey IS NULL, 'NULL_ORDER_PK', NULL),
    IF(l.l_orderkey IS NOT NULL AND o.o_orderkey IS NULL, 'INVALID_ORDER_FK', NULL),
    IF(l.l_partkey IS NULL, 'NULL_PART_PK', NULL),
    IF(l.l_partkey IS NOT NULL AND p.p_partkey IS NULL, 'INVALID_PART_FK', NULL),
    IF(l.l_suppkey IS NULL, 'NULL_SUPP_PK', NULL),
    IF(l.l_suppkey IS NOT NULL AND s.s_suppkey IS NULL, 'INVALID_SUPP_FK', NULL),
    IF(l.l_quantity <= 0, 'QTY_NOT_POSITIVE', NULL),
    IF(l.l_extendedprice <= 0, 'PRICE_NOT_POSITIVE', NULL),
    IF(l.l_discount < 0 OR l.l_discount > 0.1, 'DISCOUNT_OUT_OF_RANGE', NULL),
    IF(l.l_tax < 0 OR l.l_tax > 0.08, 'TAX_OUT_OF_RANGE', NULL),
    IF(l.l_shipdate < '1992-01-01' OR l.l_shipdate > '1998-12-31', 'SHIPDATE_OUT_OF_RANGE', NULL),
    IF(l.l_commitdate < '1992-01-01' OR l.l_commitdate > '1998-12-31', 'COMMITDATE_OUT_OF_RANGE', NULL),
    IF(l.l_receiptdate < '1992-01-01' OR l.l_receiptdate > '1998-12-31', 'RECEIPTDATE_OUT_OF_RANGE', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.lineitem l
LEFT JOIN (SELECT DISTINCT o_orderkey FROM DDEV.silver.orders) o
  ON l.l_orderkey = o.o_orderkey
LEFT JOIN (SELECT DISTINCT p_partkey FROM DDEV.silver.part) p
  ON l.l_partkey = p.p_partkey
LEFT JOIN (SELECT DISTINCT s_suppkey FROM DDEV.silver.supplier) s
  ON l.l_suppkey = s.s_suppkey;

-- Step 3: Create Silver lineitem
CREATE OR REPLACE TABLE DDEV.silver.lineitem AS
SELECT l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
FROM lineitem_validation
WHERE rejection_reason IS NULL ;

-- Step 4: Create lineitem quarantine table
CREATE OR REPLACE TABLE DDEV.silver.lineitem_quarantine AS
SELECT l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment,rejection_reason
FROM lineitem_validation
WHERE rejection_reason IS NOT NULL;



In [0]:
CREATE OR REPLACE TEMP VIEW partsupp_validation AS
SELECT ps.*,
  nullif(concat_ws(',',
    IF(ps.ps_partkey IS NULL, 'NULL_PART_PK', NULL),
    IF(ps.ps_suppkey IS NULL, 'NULL_SUPP_PK', NULL),
    IF(ps.ps_partkey IS NOT NULL AND p.p_partkey IS NULL, 'INVALID_PART_FK', NULL),
    IF(ps.ps_suppkey IS NOT NULL AND s.s_suppkey IS NULL, 'INVALID_SUPP_FK', NULL),
    IF(ps.ps_availqty IS NULL, 'MISSING_QTY', NULL),
    IF(ps.ps_availqty < 0, 'QTY_NOT_POSITIVE', NULL),
    IF(ps.ps_supplycost IS NULL, 'MISSING_SUPPLYCOST', NULL),
    IF(ps.ps_supplycost < 0, 'SUPPLYCOST_NOT_POSITIVE', NULL)
  ), '') AS rejection_reason
FROM DDEV.bronze.partsupp ps
LEFT JOIN (SELECT DISTINCT p_partkey FROM DDEV.silver.part) p
  ON ps.ps_partkey = p.p_partkey
LEFT JOIN (SELECT DISTINCT s_suppkey FROM DDEV.silver.supplier) s
  ON ps.ps_suppkey = s.s_suppkey;

-- Step 3: Create Silver partsupp
CREATE OR REPLACE TABLE DDEV.silver.partsupp AS
SELECT ps_partkey,
       ps_suppkey,
       ps_availqty,
       ps_supplycost,
       ps_comment
FROM partsupp_validation
WHERE rejection_reason IS NULL;

-- Step 4: Create partsupp quarantine table
CREATE OR REPLACE TABLE DDEV.silver.partsupp_quarantine AS
SELECT  ps_partkey,
       ps_suppkey,
       ps_availqty,
       ps_supplycost,
       ps_comment,rejection_reason
FROM partsupp_validation
WHERE rejection_reason IS NOT NULL;



In [0]:
-- Data Quality: Automated row count verification for all Bronze/Silver/Quarantine tables
CREATE OR REPLACE TEMP VIEW dq_counts AS
WITH counts AS (
  SELECT 'region'      AS table_name, COUNT(*) AS bronze FROM DDEV.bronze.region
  UNION ALL SELECT 'nation',    COUNT(*) FROM DDEV.bronze.nation
  UNION ALL SELECT 'customer',  COUNT(*) FROM DDEV.bronze.customer
  UNION ALL SELECT 'supplier',  COUNT(*) FROM DDEV.bronze.supplier
  UNION ALL SELECT 'part',      COUNT(*) FROM DDEV.bronze.part
  UNION ALL SELECT 'orders',    COUNT(*) FROM DDEV.bronze.orders
  UNION ALL SELECT 'lineitem',  COUNT(*) FROM DDEV.bronze.lineitem
  UNION ALL SELECT 'partsupp',  COUNT(*) FROM DDEV.bronze.partsupp
),
silver AS (
  SELECT 'region'      AS table_name, COUNT(*) AS silver FROM DDEV.silver.region
  UNION ALL SELECT 'nation',    COUNT(*) FROM DDEV.silver.nation
  UNION ALL SELECT 'customer',  COUNT(*) FROM DDEV.silver.customer
  UNION ALL SELECT 'supplier',  COUNT(*) FROM DDEV.silver.supplier
  UNION ALL SELECT 'part',      COUNT(*) FROM DDEV.silver.part
  UNION ALL SELECT 'orders',    COUNT(*) FROM DDEV.silver.orders
  UNION ALL SELECT 'lineitem',  COUNT(*) FROM DDEV.silver.lineitem
  UNION ALL SELECT 'partsupp',  COUNT(*) FROM DDEV.silver.partsupp
),
quarantine AS (
  SELECT 'region'      AS table_name, COUNT(*) AS quarantine FROM DDEV.silver.region_quarantine
  UNION ALL SELECT 'nation',    COUNT(*) FROM DDEV.silver.nation_quarantine
  UNION ALL SELECT 'customer',  COUNT(*) FROM DDEV.silver.customer_quarantine
  UNION ALL SELECT 'supplier',  COUNT(*) FROM DDEV.silver.supplier_quarantine
  UNION ALL SELECT 'part',      COUNT(*) FROM DDEV.silver.part_quarantine
  UNION ALL SELECT 'orders',    COUNT(*) FROM DDEV.silver.orders_quarantine
  UNION ALL SELECT 'lineitem',  COUNT(*) FROM DDEV.silver.lineitem_quarantine
  UNION ALL SELECT 'partsupp',  COUNT(*) FROM DDEV.silver.partsupp_quarantine
)
SELECT c.table_name, c.bronze, s.silver, q.quarantine,
       (c.bronze = s.silver + q.quarantine) AS counts_match
FROM counts c
JOIN silver s ON c.table_name = s.table_name
JOIN quarantine q ON c.table_name = q.table_name;
